# AC optimal power flow in rectangular coordinates

The **optimal power flow (OPF)** problem chooses generator setpoints and bus
voltages to meet electrical demand at minimum generation cost, subject to the
physics of the network. It was introduced by {cite:t}`Carpentier1962` and has
remained the workhorse formulation of power-system operations and planning ever
since {cite:p}`FrankSteponavice2012`.

The hard variant is **AC-OPF**, which enforces the full nonlinear AC power-flow
(Kirchhoff) equations rather than a linearized DC approximation. At every bus the
net injected complex power must equal generation minus demand, voltage magnitudes
must stay within $[V_{\min}, V_{\max}]$, and generation must respect its real and
reactive limits. Because the power injections are quadratic in the bus voltages,
AC-OPF is a **nonconvex quadratically-constrained quadratic program (QCQP)** — it
is NP-hard in general and is the canonical benchmark for relaxation and global
optimization research in power systems {cite:p}`Molzahn2019`.

discopt's `discopt.opf` module builds AC-OPF directly as a discopt
`Model`, so the spatial branch-and-bound solver handles it as an ordinary QCQP.

## The rectangular-voltage formulation

discopt writes each bus voltage phasor in **rectangular coordinates**,
$V_i = e_i + j\,f_i$, rather than the more common polar magnitude/angle form. The
rectangular form is the natural one for global optimization: the power-balance
equations become *polynomial* (in fact purely quadratic) in $(e, f)$, with no
transcendental $\sin/\cos$ terms {cite:p}`Molzahn2019`.

With the bus admittance matrix $Y = G + jB$ assembled from the line series
impedances $z = r + jx$, the injected real and reactive power at bus $i$ are

$$
P_i = \sum_k \big[\, G_{ik}(e_i e_k + f_i f_k) + B_{ik}(f_i e_k - e_i f_k) \,\big],
$$

$$
Q_i = \sum_k \big[\, G_{ik}(f_i e_k - e_i f_k) - B_{ik}(e_i e_k + f_i f_k) \,\big].
$$

The model then imposes

- **power balance** (bilinear equalities): $P_i = P^g_i - P^d_i$ and
  $Q_i = Q^g_i - Q^d_i$ at every bus;
- **voltage magnitude limits** (quadratic): $V_{\min}^2 \le e_i^2 + f_i^2 \le V_{\max}^2$;
- a **slack/reference bus** that pins the angle by fixing $f_i = 0$ and $e_i$ to a
  reference value;
- **generator box limits** $P^g_{\min}\!\le\! P^g \!\le\! P^g_{\max}$ (and likewise for $Q^g$).

The objective minimizes linear generation cost $\sum_i c_i\, P^g_i$. Every
constraint is linear or quadratic and the bilinear products $e_i e_k$, $f_i f_k$,
$f_i e_k$ make the equalities nonconvex — exactly the QCQP structure that discopt's
spatial branch-and-bound, with its convex (SOC/PSD) relaxations of the bilinear
terms, is built to certify {cite:p}`Jabr2006,Molzahn2019`.

## The built-in two-bus example

`two_bus_example()` returns the smallest meaningful AC-OPF: a slack generator at
bus `g` feeds a PQ load (demand $P^d = 0.5$, $Q^d = 0.2$ p.u.) at bus `l` across a
single lossy line ($z = 0.01 + j\,0.1$). We build the rectangular QCQP and solve
it to global optimality. The cost is the real power the generator must produce —
the load plus the resistive line loss.

In [1]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_ENABLE_X64"] = "1"

import numpy as np

from discopt.opf import build_ac_opf_rectangular, two_bus_example

ac = two_bus_example()
model = build_ac_opf_rectangular(ac)
result = model.solve()

x = result.x
print(f"status        : {result.status}")
print(f"cost (gen Pg) : {float(result.objective):.4f}  p.u.")
print(f"B&B nodes     : {result.node_count}")
print()
for b in ac.buses:
    e, f = float(x[f"e_{b.name}"]), float(x[f"f_{b.name}"])
    vmag = np.hypot(e, f)
    print(f"bus {b.name:>2} ({b.kind:>5}) : |V| = {vmag:.4f}  angle = {np.degrees(np.arctan2(f, e)):+.2f} deg")
print()
print(f"slack dispatch: Pg = {float(x['pg_g']):.4f}  Qg = {float(x['qg_g']):.4f}  p.u.")

status        : optimal
cost (gen Pg) : 0.5031  p.u.
B&B nodes     : 1

bus  g (slack) : |V| = 1.0000  angle = +0.00 deg
bus  l ( load) : |V| = 0.9731  angle = -2.83 deg

slack dispatch: Pg = 0.5031  Qg = 0.2306  p.u.


## Building a custom network

Any small network is specified directly from `Bus` and `Line` dataclasses. Here we
build a **three-bus** system with two competing generators serving one load: a
cheap slack generator `g1` (cost 1) and an expensive generator `g2` (cost 3). The
economic dispatch should lean entirely on the cheap unit, with `g2` left idle, so
long as the line and voltage limits permit it.

In [2]:
from discopt.opf import ACOPF, Bus, Line, admittance_matrix

ac3 = ACOPF(
    buses=[
        Bus("g1", kind="slack", pg_max=2.0, qg_min=-1.0, qg_max=1.0, cost=1.0),  # cheap
        Bus("g2", kind="gen", pg_max=1.5, qg_min=-1.0, qg_max=1.0, cost=3.0),    # expensive
        Bus("load", kind="load", pd=1.0, qd=0.3),
    ],
    lines=[
        Line("g1", "load", r=0.02, x=0.2),
        Line("g2", "load", r=0.02, x=0.2),
        Line("g1", "g2", r=0.01, x=0.1),
    ],
)

G, B = admittance_matrix(ac3)
print(f"3x3 bus admittance matrix assembled (G + jB), conductance G =\n{np.round(G, 2)}\n")

res3 = build_ac_opf_rectangular(ac3).solve()
x3 = res3.x
print(f"status : {res3.status}")
print(f"cost   : {float(res3.objective):.4f}  p.u.")
print()
for b in ac3.buses:
    if b.kind in ("slack", "gen"):
        print(f"gen {b.name:>2} (cost {b.cost:.0f}) : Pg = {float(x3['pg_' + b.name]):.4f}  p.u.")

3x3 bus admittance matrix assembled (G + jB), conductance G =
[[ 1.49 -0.99 -0.5 ]
 [-0.99  1.49 -0.5 ]
 [-0.5  -0.5   0.99]]



status : optimal
cost   : 1.0141  p.u.

gen g1 (cost 1) : Pg = 1.0141  p.u.
gen g2 (cost 3) : Pg = 0.0000  p.u.


As expected, the cheap generator `g1` carries the full load (plus losses) while
the expensive `g2` stays at zero real dispatch — the merit-order solution to this
small economic dispatch, certified globally optimal through the nonconvex AC
constraints.

## Why AC-OPF is a capstone test problem

AC-OPF is the canonical, *application-grade* nonconvex QCQP, which makes it the
capstone validation target for discopt's QCQP relaxation work. The bilinear
power-balance equalities are precisely the structure addressed by convex
relaxations of the power-flow equations: the **second-order-cone (SOCP)**
relaxation of {cite:t}`Jabr2006` and the broader family of SDP/SOC moment
relaxations surveyed by {cite:t}`Molzahn2019`. By lifting the products
$e_i e_k, f_i f_k, f_i e_k$ into a matrix variable and imposing PSD/SOC cuts,
these relaxations provide tight lower bounds that often close the optimality gap
at, or near, the root node.

discopt's rectangular formulation feeds this machinery unchanged: the same spatial
branch-and-bound that solves the pooling and bilinear benchmarks tackles AC-OPF as
an ordinary QCQP, and the PSD/SOC cuts that strengthen its bilinear envelopes are
exactly what a real power network needs to certify global optimality
{cite:p}`Jabr2006,Molzahn2019`.